In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
from src.preprocessing import get_preprocessor 

df = pd.read_csv("Ames_Housing_Processed.csv")


X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model_lgbm = LGBMRegressor(n_estimators=100)

param_grid = {
    'n_estimators': [1000, 2000],          # Yeterli öğrenme kapasitesi
    'learning_rate': [0.01, 0.03],              # Dengeli hız
    'num_leaves': [31, 63, 127],               # Karmaşıklığı kontrol eden ana düğme
    'max_depth': [-1, 10, 15],                  # Yaprak büyümesine tavan sınırı
    'colsample_bytree': [0.6, 0.8],            # Özellik gürültüsünü azaltma
    'min_child_samples': [20],             # Aşırı dallanmayı önleyen bariyer
    'reg_alpha': [0.1, 0.5],              # L1 regülarizasyonu ile overfitting freni
    'reg_lambda': [0.1, 0.5]              # L2 regülarizasyonu ile stabilite
}


grid_search = GridSearchCV(estimator=model_lgbm, param_grid=param_grid, cv=5,  scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Params", grid_search.best_params_)

y_pred = grid_search.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print(f"R2: {r2}")
print(f"MAE: {mae}")
print(f"RMSE: {rmse}")

Best Params {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 1000, 'num_leaves': 31, 'reg_alpha': 0.5, 'reg_lambda': 0.1}
R2: 0.9102498416699996
MAE: 13264.774402786745
RMSE: 20547.061704833104
